# SI Figure — factorised vs. non-factorised image-formation model

Addresses Reviewer 4, Major Comment 5's second sub-issue (see `claude/SI_FactorsationofModel.md`
for the full plan and `claude/SI_ImageModel.md` for the first sub-issue, already fixed in
`SI.tex`). The production simulation factorises the wavelength dependence: a single
mean-wavelength PSF width $\sigma(\bar\lambda)$ for every sampled photon, and a separately
spectrum-averaged detector response $QE_{d,n}=\int p(\lambda)QE_n(\lambda)d\lambda$. This
notebook compares that production model (A) against two increasingly rigorous references: (B)
a non-factorised, photon-by-photon model (each photon gets its own wavelength-dependent PSF
width *and* detection probability, but the PSF is still a Gaussian), and (C) a true
diffraction-based reference built from `VectorialPSF`'s established vectorial Debye/
Richards-Wolf optics — the reviewer's stronger "ideally diffraction-based" requirement. All
three run across the same photon-count range used by this repo's other simulation notebooks
(e.g. `Figure_1c.ipynb`, `FigureSI_SBR.ipynb`). The headline result is whether the production
model (A) stays close to the true optics reference (C).

**`FAST_MODE` flag (below):** `True` (default) runs a small, fast grid (fewer photon levels,
300 repeats/condition, fewer wavelength bins for model C) that reproduces the same trend in
well under a minute or two. `FAST_MODE = False` reproduces the full published comparison
(`N_REPEATS=20000`, matching/exceeding the `\ref{note:tradeoffs}` precedent of 10,000) —
genuinely minutes of compute, since models (B) and (C) are substantially slower than (A) per
condition.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s — %(levelname)s — %(message)s',
)

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# Bare `pyS3M.X` imports -- see notebooks/figures/Figure_1c.ipynb for why (Enum/dataclass
# identity consistency with the fitting pipeline's own imports).
from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions, PlottingBase
from pyS3M.PSFFunctions import PSF_Functions
from pyS3M.simulation.multicolour import MultiC_Sim_Funcs

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
PSF_F = PSF_Functions()
MSF = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

REPO_ROOT = Path.cwd().resolve().parents[2]  # notebooks/figures/SI -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT = Path("outputs") / "FigureSI_Factorisation"  # git-ignored
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Model (B) costs scale with N_REPEATS x n_photons (each photon gets its own array entry),
# not just N_REPEATS -- much steeper than the other FAST_MODE sweeps in this repo, so both
# axes are cut for FAST_MODE, not just the bootstrap count.
FAST_MODE = True  # False -> the full N_REPEATS=20000 published comparison
print(f"FAST_MODE={FAST_MODE}")

## Conditions

Matches the plan in `claude/SI_FactorsationofModel.md` §3/§6: the same three dyes, camera,
NA, pixel size and background already used for these exact dyes in
`\ref{note:tradeoffs}` (Ximea-vs-Prime95B comparison) -- not fresh parameters.

In [ ]:
dyes = ['ATTO 488', 'ATTO 565', 'ATTO 647N']

N_REPEATS = 300 if FAST_MODE else 20000
N_PHOTON_STEPS = 8 if FAST_MODE else 15

# Same 500-50,000 log-spaced range/formula as this repo's other simulation-sweep notebooks
# (e.g. FigureSI_SBR.ipynb), not a bespoke set of levels.
photon_levels = np.unique(np.around(np.logspace(np.log10(500), np.log10(50000), N_PHOTON_STEPS) / 5) * 5)

# Sub-pixel offsets from the ROI centre, as a fraction of one pixel -- diagonal-only subset
# of the full Bayer unit cell (see plan §3/§6)
offsets_frac = [(0.0, 0.0), (0.25, 0.25), (0.5, 0.5)]

BACKGROUND_PHOTONS = 10.0   # ph/px, matches \ref{note:tradeoffs}
BACKGROUND_COLOUR = np.array([1.0, 1.0, 1.0])
NA = 1.49
PIXEL_SIZE_NM = 69.0        # Ximea
n_unit_cells = 7            # -> 14x14 px ROI, same convention as Figure_1c.ipynb

RNG_SEED = 12345

print(f"{len(dyes)} dyes x {len(photon_levels)} photon levels x {len(offsets_frac)} offsets "
      f"x N_REPEATS={N_REPEATS} = {len(dyes)*len(photon_levels)*len(offsets_frac)*N_REPEATS} "
      f"draws per model")

## Camera calibration & Bayer layout

Identical setup to `Figure_1c.ipynb` -- same calibration files, same `MaskFunctions.get_masks`
call, same `pixel_order_indices` convention. Both model (A) and model (B) below read the Bayer
colour of a given pixel from this *same* `masks`/`pixel_order_indices` pair, so the two models
can't disagree because of two different Bayer-layout definitions -- only because of the
factorisation question itself.

In [ ]:
gain = IO.read_tiff(str(CALIBRATION_DIR / "gain.tif"))
offset = IO.read_tiff(str(CALIBRATION_DIR / "offset.tif"))
rqe = IO.read_tiff(str(CALIBRATION_DIR / "rqe.tif"))

gain_median = float(np.median(gain))
offset_median = float(np.median(offset))
rqe_median = float(np.median(rqe))

R_qe, G_qe, B_qe, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_qe, G_qe, R_qe])  # B=0, G=1, R=2 -- matches pixel_order below
pixel_order = ['B', 'G', 'R']
pixel_order_indices = {'B': 0, 'G': 1, 'R': 2}
qe_by_channel = [B_qe, G_qe, R_qe]  # same order as pixel_order_indices

H = W = 2 * n_unit_cells
masks = M_F.get_masks(size_x=H, size_y=W)

# Single integer colour-index grid, shared by both models (see markdown above)
colour_grid = np.full((H, W), -1, dtype=int)
for c, idx in pixel_order_indices.items():
    colour_grid[masks[c]] = idx

camera_parameters_base = {
    'gain':                np.full((H, W), gain_median),
    'offset':              np.full((H, W), offset_median),
    'variance':            np.full((H, W), 1.0),
    'readnoise':           1.0,
    'rqe':                 np.full((H, W), rqe_median),
    'masks':               masks,
    'pixel_QYs':           pixel_QYs,
    'pixel_order':         pixel_order,
    'pixel_order_indices': pixel_order_indices,
    'mosaic_unit':         M_F.mosaic_unit,
}

x0_centre_nm = PIXEL_SIZE_NM * H / 2.0
y0_centre_nm = PIXEL_SIZE_NM * W / 2.0
print(f"ROI: {H}x{W} px, centre at ({x0_centre_nm:.1f}, {y0_centre_nm:.1f}) nm")

## Shared helpers: background rate & moment-based sigma

Factored out so model (A) and (B) images get identically background-subtracted before the sigma_PSF estimate below -- a moment (weighted second-moment) estimate, not the real nonlinear Gaussian fit used in production (that's still deferred -- see 'Next steps' at the bottom), but fast and unbiased enough to compare the two *models* against each other.

In [ ]:
# Per-pixel background rate (photoelectrons), matching what the production path's
# QE-thinning trick converges to for a single dye (see simulate_nonfactorised's comment
# below) -- factored out here so both models' background can be identically subtracted
# when estimating a moment-based sigma_PSF below.
def background_rate_grid(background_photons, background_colour, masks, pixel_order_indices, H, W, n_dyes=1):
    bg_perdye = background_photons / n_dyes
    bg_colour_norm = np.asarray(background_colour) / np.sum(background_colour)
    bg_rate = np.zeros((H, W))
    for c, idx in pixel_order_indices.items():
        bg_rate[masks[c]] = bg_colour_norm[idx] * bg_perdye
    return bg_rate


# Photon-weighted second-moment estimate of PSF width, per repeat -- a fast proxy for the
# recovered sigma a real nonlinear fit would report (that fit is still deferred, see 'Next
# steps'). Background-subtracted first (moment estimates are sensitive to an unsubtracted
# pedestal, especially at low photon counts where background is a large fraction of the
# signal); negative post-subtraction values are clipped to zero weight. Isotropic sigma via
# sigma^2 = (var_x + var_y) / 2 (exact for a circular 2D Gaussian).
def moment_sigma_nm(image_stack, x0_px, y0_px, bg_rate):
    sig = np.clip(image_stack.astype(float) - bg_rate[None, :, :], 0, None)
    Hh, Ww = image_stack.shape[1:]
    yy, xx = np.indices((Hh, Ww))
    total = sig.sum(axis=(1, 2))
    total_safe = np.maximum(total, 1e-9)
    var_x = (sig * (xx[None] - x0_px) ** 2).sum(axis=(1, 2)) / total_safe
    var_y = (sig * (yy[None] - y0_px) ** 2).sum(axis=(1, 2)) / total_safe
    sigma_px = np.sqrt(np.maximum((var_x + var_y) / 2.0, 0))
    return sigma_px * PIXEL_SIZE_NM

## Model (A) — factorised, the real production code

Not reimplemented: calls `MultiC_Sim_Funcs.gen_camera_image_stack` directly, the same function
the rest of the paper's simulations use (via `test_simulation_method_2d_sweep`), with
`return_photoelectrons_stack=True` to get photoelectrons per pixel straight after the Binomial
QE step (Eq. S8) -- before gain/offset/read-noise, which are identical, uninteresting noise
applied downstream of the question this notebook is about.

In [ ]:
def simulate_factorised(dye, n_photons_val, x0_nm, y0_nm, n_repeats, camera_parameters):
    '''Model (A): the real, production factorised simulation path.'''
    avg_wl, dye_pixel_efficiency = S_F.get_pixel_fractions_dye_and_filters(
        [dye], [], wavelength, camera_parameters['pixel_QYs'], normalized=False,
    )
    x0y0 = {"dye": np.zeros([n_repeats, 2, 1])}
    x0y0["dye"][:, 0, 0] = y0_nm  # row axis = y
    x0y0["dye"][:, 1, 0] = x0_nm  # col axis = x
    n_photons = {"dye": np.full(n_repeats, n_photons_val, dtype=float)}

    pe_stack = MSF.gen_camera_image_stack(
        camera_parameters, wavelength, avg_wl, dye_pixel_efficiency, n_photons, x0y0,
        smoothing_function=None,
        background_photons=BACKGROUND_PHOTONS,
        background_colour=list(BACKGROUND_COLOUR),
        NA=NA, pixel_size=PIXEL_SIZE_NM,
        return_photoelectrons_stack=True,
    )
    return pe_stack  # (n_repeats, H, W) int32

## Model (B) — non-factorised, photon-by-photon reference

New implementation, per the reviewer's suggested joint form. For each photon: its own sampled
wavelength gives *both* its own PSF width (spatial placement) *and* its own per-channel
detection probability (no spectrum-averaging of either). Fully vectorised over
(repeat, photon) -- no Python-level loop over either axis, so this should already be about as
fast as a pure-numpy implementation gets; if it's still too slow at scale the next step would
be numba, not restructuring this.

In [ ]:
def _sample_wavelengths(spectrum, wl, shape, rng):
    '''Vectorised inverse-CDF sampling -- same method as
    SpectralFunctions.sample_photons_from_spectrum, generalised to an arbitrary output shape.'''
    pdf = np.maximum(spectrum, 0)
    pdf = pdf / np.trapz(pdf, wl)
    dx = np.diff(wl)
    pdf_avg = 0.5 * (pdf[1:] + pdf[:-1])
    cdf = np.zeros(len(wl))
    cdf[1:] = np.cumsum(pdf_avg * dx)
    cdf = cdf / cdf[-1]
    u = rng.random(shape)
    return np.interp(u.ravel(), cdf, wl).reshape(shape)


def simulate_nonfactorised(dye, n_photons_val, x0_nm, y0_nm, n_repeats, camera_parameters, rng):
    '''Model (B): joint wavelength-resolved reference, photon-by-photon.'''
    n_photons_int = int(n_photons_val)
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]

    shape = (n_repeats, n_photons_int)
    lam = _sample_wavelengths(dye_spectrum, wavelength, shape, rng)          # own wavelength...
    sigma_px = PSF_F.sigma_PSF(lam, NA) / PIXEL_SIZE_NM                       # ...own PSF width

    x0_px, y0_px = x0_nm / PIXEL_SIZE_NM, y0_nm / PIXEL_SIZE_NM
    xk = x0_px + rng.standard_normal(shape) * sigma_px
    yk = y0_px + rng.standard_normal(shape) * sigma_px
    ik = np.round(xk).astype(np.int64)  # column index
    jk = np.round(yk).astype(np.int64)  # row index

    H, W = camera_parameters['gain'].shape
    valid = (ik >= 0) & (ik < W) & (jk >= 0) & (jk < H)
    colour_k = np.full(shape, -1, dtype=np.int64)
    colour_k[valid] = colour_grid[jk[valid], ik[valid]]

    QE_k = np.zeros(shape)                                                    # ...own detection prob.
    for c_idx, curve in enumerate(qe_by_channel):
        sel = colour_k == c_idx
        if np.any(sel):
            QE_k[sel] = np.interp(lam[sel], wavelength, curve)

    detected = valid & (rng.random(shape) < QE_k)

    r_idx = np.broadcast_to(np.arange(n_repeats)[:, None], shape)
    flat_idx = (r_idx * H + jk) * W + ik
    counts = np.bincount(flat_idx[detected], minlength=n_repeats * H * W)
    image = counts.reshape(n_repeats, H, W).astype(np.int32)

    # Background: added as photoelectrons directly at the rate background_rate_grid gives --
    # algebraically what the production path's raw-photon-then-QE-thin trick
    # (gen_camera_image_stack's `background_photons_matrix`, which pre-divides by
    # dye_pixel_efficiency so that *after* Binomial(QE) thinning the photoelectron background
    # lands on `background_photons`) also converges to, for a single dye. Kept simple here
    # since it's the same end result, not a new assumption. Shared with the moment-sigma
    # estimate above so both use an identical background definition.
    bg_rate = background_rate_grid(BACKGROUND_PHOTONS, BACKGROUND_COLOUR, masks, pixel_order_indices, H, W)
    image = image + rng.poisson(bg_rate, size=(n_repeats, H, W)).astype(np.int32)

    return image

## Model (C) — true diffraction-based reference (VectorialPSF)

Both (A) and (B) above share the same Gaussian PSF-width approximation
(`PSF_Functions.sigma_PSF`, σ = λ/(√2·π·NA)) — (B) only wavelength-resolves *which* σ each
photon gets, it doesn't change the Gaussian functional form itself. Per the reviewer's
stronger "ideally" clause, this builds a third reference that replaces the Gaussian entirely
with a genuine vectorial Debye/Richards-Wolf diffraction PSF
(`simulation/defocus_psf.py::VectorialPSF` — Fourier-optics propagation of the apodised pupil
field). Established equations, not an approximation of an approximation — and not new,
unvalidated code: this class was already built and unit-tested (16 tests: normalisation,
in-focus symmetry, defocus broadening, spherical-aberration symmetry-breaking, lateral shift,
aperture clipping) for this project's Z-defocus simulation work; this notebook is its first
use as a validation reference for the in-focus factorisation question.

**Computational note**: an FFT-based diffraction patch for one (wavelength, sub-pixel offset)
pair is too expensive to recompute per individual photon at its own exact continuous
wavelength. Instead, each dye's emission spectrum is discretised into `N_WAVELENGTH_BINS`
representative wavelengths (stratified by spectral probability mass, not evenly spaced), and
the diffraction patch is precomputed once per (dye, offset, bin) — reused across every
photon_level/repeat below, since the *shape* of the diffraction pattern only depends on
wavelength/offset, not on how many photons are drawn from it. Each photon's own continuous
wavelength is snapped to its nearest bin for spatial placement only; detection QE still uses
the continuous value, unchanged from model (B).

In [ ]:
from pyS3M.simulation.defocus_psf import VectorialPSF

N_WAVELENGTH_BINS = 8 if FAST_MODE else 15


def build_wavelength_bins(spectrum, wl, n_bins):
    '''n_bins representative wavelengths + bin edges, stratified by the dye's own
    emission-spectrum CDF (equal probability mass per bin, not equal wavelength spacing).'''
    pdf = np.maximum(spectrum, 0)
    pdf = pdf / np.trapz(pdf, wl)
    dx = np.diff(wl)
    pdf_avg = 0.5 * (pdf[1:] + pdf[:-1])
    cdf = np.zeros(len(wl))
    cdf[1:] = np.cumsum(pdf_avg * dx)
    cdf = cdf / cdf[-1]
    edges_p = np.linspace(0, 1, n_bins + 1)
    edges_wl = np.interp(edges_p, cdf, wl)
    centres_p = (edges_p[:-1] + edges_p[1:]) / 2
    centres_wl = np.interp(centres_p, cdf, wl)
    return centres_wl, edges_wl


vpsf = VectorialPSF(NA=NA, n_medium=1.33, n_immersion=1.515, pix_obj_um=PIXEL_SIZE_NM / 1000.0)
roi_centre_row = int(round(y0_centre_nm / PIXEL_SIZE_NM))
roi_centre_col = int(round(x0_centre_nm / PIXEL_SIZE_NM))

diffraction_cache = {}  # (dye, offset) -> (patches (n_bins, ps, ps), bin_edges_wl)
t0 = time.time()
for dye in dyes:
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]
    bin_centres_wl, bin_edges_wl = build_wavelength_bins(dye_spectrum, wavelength, N_WAVELENGTH_BINS)
    for (fx, fy) in offsets_frac:
        patches = vpsf.compute_psf_stack(
            z_offsets_um=np.array([0.0]),
            wavelengths_um=bin_centres_wl / 1000.0,
            x0_um=fx * (PIXEL_SIZE_NM / 1000.0),
            y0_um=fy * (PIXEL_SIZE_NM / 1000.0),
        )[0]  # (N_WAVELENGTH_BINS, psf_size, psf_size)
        diffraction_cache[(dye, (fx, fy))] = (patches, bin_edges_wl)

print(f"Precomputed {len(diffraction_cache)} (dye, offset) diffraction PSF sets "
      f"({N_WAVELENGTH_BINS} wavelength bins each, psf_size={vpsf.psf_size}) "
      f"in {time.time()-t0:.1f}s")

In [ ]:
def simulate_diffraction(dye, n_photons_val, n_repeats, camera_parameters, rng, offset):
    '''Model (C): true vectorial-diffraction reference, photon-by-photon. Spatial placement
    draws from a precomputed VectorialPSF patch (nearest wavelength bin); detection QE uses
    each photon's own continuous sampled wavelength, unchanged from model (B).'''
    n_photons_int = int(n_photons_val)
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]
    patches, bin_edges_wl = diffraction_cache[(dye, offset)]

    shape = (n_repeats, n_photons_int)
    lam = _sample_wavelengths(dye_spectrum, wavelength, shape, rng)
    bin_idx = np.clip(np.digitize(lam, bin_edges_wl) - 1, 0, len(patches) - 1)

    H, W = camera_parameters['gain'].shape
    ps = patches.shape[1]
    patch_centre = ps // 2

    ik = np.zeros(shape, dtype=np.int64)
    jk = np.zeros(shape, dtype=np.int64)
    for b in range(len(patches)):
        sel = bin_idx == b
        n_sel = int(sel.sum())
        if n_sel == 0:
            continue
        flat_p = patches[b].ravel()
        flat_p = flat_p / flat_p.sum()
        draw = rng.choice(flat_p.size, size=n_sel, p=flat_p)
        r, c = np.unravel_index(draw, patches[b].shape)
        jk[sel] = roi_centre_row - patch_centre + r
        ik[sel] = roi_centre_col - patch_centre + c

    valid = (ik >= 0) & (ik < W) & (jk >= 0) & (jk < H)
    colour_k = np.full(shape, -1, dtype=np.int64)
    colour_k[valid] = colour_grid[jk[valid], ik[valid]]

    QE_k = np.zeros(shape)
    for c_idx, curve in enumerate(qe_by_channel):
        sel = colour_k == c_idx
        if np.any(sel):
            QE_k[sel] = np.interp(lam[sel], wavelength, curve)

    detected = valid & (rng.random(shape) < QE_k)

    r_idx = np.broadcast_to(np.arange(n_repeats)[:, None], shape)
    flat_idx = (r_idx * H + jk) * W + ik
    counts = np.bincount(flat_idx[detected], minlength=n_repeats * H * W)
    image = counts.reshape(n_repeats, H, W).astype(np.int32)

    bg_rate = background_rate_grid(BACKGROUND_PHOTONS, BACKGROUND_COLOUR, masks, pixel_order_indices, H, W)
    image = image + rng.poisson(bg_rate, size=(n_repeats, H, W)).astype(np.int32)

    return image

## Run all three models

Runs (A) factorised, (B) photon-by-photon (still Gaussian), and (C) the true diffraction-based
reference (`VectorialPSF`) at the `N_REPEATS`/`photon_levels` set above across every (dye,
photon level, offset) condition.

In [ ]:
results = {}  # (dye, n_photon, offset) -> {'A': image, 'B': image, 'C': image}

rng_B = np.random.default_rng(RNG_SEED)
rng_C = np.random.default_rng(RNG_SEED + 1)

t_start = time.time()
for dye in dyes:
    for n_photon in photon_levels:
        for (fx, fy) in offsets_frac:
            x0_nm = x0_centre_nm + fx * PIXEL_SIZE_NM
            y0_nm = y0_centre_nm + fy * PIXEL_SIZE_NM
            key = (dye, float(n_photon), (fx, fy))

            img_A = simulate_factorised(dye, n_photon, x0_nm, y0_nm, N_REPEATS, camera_parameters_base)
            img_B = simulate_nonfactorised(dye, n_photon, x0_nm, y0_nm, N_REPEATS, camera_parameters_base, rng_B)
            img_C = simulate_diffraction(dye, n_photon, N_REPEATS, camera_parameters_base, rng_C, (fx, fy))

            results[key] = {'A': img_A, 'B': img_B, 'C': img_C}

total_time = time.time() - t_start
n_conditions = len(dyes) * len(photon_levels) * len(offsets_frac)
print(f"{n_conditions} conditions x N_REPEATS={N_REPEATS}, total wall time {total_time/60:.1f} min")

## Visual comparison

Mean image over the `N_REPEATS` repeats for (A) factorised, (B) photon-by-photon (still
Gaussian), and (C) the true diffraction reference, for one representative condition per dye
(middle photon level, zero offset). The right-most panel is (A) − (C) — the actual quantity of
interest: how far the production model departs from the true optics.

In [ ]:
mid_photon = photon_levels[len(photon_levels) // 2]
fig, axs = plotter.two_column_plot(nrows=len(dyes), ncols=4, width=6.69, height=6.0)

for row, dye in enumerate(dyes):
    key = (dye, float(mid_photon), (0.0, 0.0))
    mean_A = results[key]['A'].mean(axis=0)
    mean_B = results[key]['B'].mean(axis=0)
    mean_C = results[key]['C'].mean(axis=0)
    diff = mean_A - mean_C
    vmax = max(mean_A.max(), mean_B.max(), mean_C.max())

    for col, (img, title, cmap, vmin, vm) in enumerate([
        (mean_A, f'{dye}\n(A) factorised', 'gist_gray', 0, vmax),
        (mean_B, '(B) photon-by-photon\n(Gaussian)', 'gist_gray', 0, vmax),
        (mean_C, '(C) diffraction\n(VectorialPSF)', 'gist_gray', 0, vmax),
        (diff, 'A - C', 'RdBu_r', -np.abs(diff).max(), np.abs(diff).max()),
    ]):
        ax = axs[row, col]
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vm)
        ax.set_title(title, fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, pad=0.02, fraction=0.046)

plt.savefig(OUT_ROOT / "visual_comparison.svg")
plt.show()

## Quantitative comparison

First, per-condition aggregate summary (every dye x photon-level x offset cell), framed against
(C) the true diffraction reference — then a closer look at three key differentiators
(sigma_PSF, N_photoelectrons, spectral fingerprint) as full distributions, not just means,
below.

In [ ]:
rows = []
for dye in dyes:
    for n_photon in photon_levels:
        for (fx, fy) in offsets_frac:
            key = (dye, float(n_photon), (fx, fy))
            img_A, img_B, img_C = results[key]['A'], results[key]['B'], results[key]['C']

            total_A = img_A.sum(axis=(1, 2)).astype(float)
            total_B = img_B.sum(axis=(1, 2)).astype(float)
            total_C = img_C.sum(axis=(1, 2)).astype(float)

            frac_A = {c: (img_A * (colour_grid == idx)).sum(axis=(1, 2)) / np.maximum(total_A, 1)
                      for c, idx in pixel_order_indices.items()}
            frac_B = {c: (img_B * (colour_grid == idx)).sum(axis=(1, 2)) / np.maximum(total_B, 1)
                      for c, idx in pixel_order_indices.items()}
            frac_C = {c: (img_C * (colour_grid == idx)).sum(axis=(1, 2)) / np.maximum(total_C, 1)
                      for c, idx in pixel_order_indices.items()}

            row = {
                'dye': dye, 'n_photon': n_photon, 'offset': f'({fx},{fy})',
                'total_A_mean': total_A.mean(), 'total_B_mean': total_B.mean(), 'total_C_mean': total_C.mean(),
            }
            for c in pixel_order:
                row[f'frac_{c}_A'] = frac_A[c].mean()
                row[f'frac_{c}_B'] = frac_B[c].mean()
                row[f'frac_{c}_C'] = frac_C[c].mean()
                row[f'frac_{c}_diff_AvsC_pp'] = (frac_A[c].mean() - frac_C[c].mean()) * 100
                row[f'frac_{c}_diff_BvsC_pp'] = (frac_B[c].mean() - frac_C[c].mean()) * 100
            rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUT_ROOT / "factorisation_comparison_summary.csv", index=False)
summary_df[['dye', 'n_photon', 'offset', 'total_A_mean', 'total_B_mean', 'total_C_mean',
            'frac_R_diff_AvsC_pp', 'frac_G_diff_AvsC_pp', 'frac_B_diff_AvsC_pp']]

### Distributional comparison: sigma_PSF, N_photoelectrons, spectral fingerprint

Pooled across the 3 Bayer-phase offsets, at the middle photon level, per dye (`3 x N_REPEATS`
draws per model per dye) -- enough for a real look at distribution shape, not just means.

- **sigma_PSF**: a photon-weighted second-moment estimate (background-subtracted), *not* the real nonlinear Gaussian fit used in production -- that's still deferred (see 'Next steps'). Fast and identically applied to all three models, so it's a fair way to compare them, even though it isn't the number the paper would ultimately report.
- **N_photoelectrons**: total detected photoelectrons per repeat, no fitting needed.
- **spectral fingerprint**: reduced to each dye's own dominant Bayer channel's fraction of the total (chosen automatically per dye from model (A)) -- a single representative number per repeat rather than the full 2-DOF (R,G,B) simplex, kept simple for this pass.

Each row below is one (dye, metric) pair: **left** -- (A), (B), and (C) histograms overlaid, dotted vertical lines at each distribution's mean; **right** -- how little (A) and (B) differ from the true reference (C), quantified as ECDF(A)-ECDF(C) and ECDF(B)-ECDF(C), with each curve's Kolmogorov-Smirnov statistic/p-value/percent mean difference against (C) annotated in the title.

In [ ]:
# --- Pool per-repeat metrics across the 3 offsets, at the middle photon level, per dye ---
bg_rate_grid_main = background_rate_grid(BACKGROUND_PHOTONS, BACKGROUND_COLOUR, masks, pixel_order_indices, H, W)

metrics = {}
for dye in dyes:
    key0 = (dye, float(mid_photon), offsets_frac[0])
    img_A0 = results[key0]['A']
    ch_means = {c: (img_A0 * (colour_grid == idx)).sum(axis=(1, 2)).mean()
                for c, idx in pixel_order_indices.items()}
    dom_channel = max(ch_means, key=ch_means.get)
    dom_idx = pixel_order_indices[dom_channel]

    sigma_A_list, sigma_B_list, sigma_C_list = [], [], []
    N_A_list, N_B_list, N_C_list = [], [], []
    frac_A_list, frac_B_list, frac_C_list = [], [], []

    for (fx, fy) in offsets_frac:
        key = (dye, float(mid_photon), (fx, fy))
        img_A, img_B, img_C = results[key]['A'], results[key]['B'], results[key]['C']
        x0_px = (x0_centre_nm + fx * PIXEL_SIZE_NM) / PIXEL_SIZE_NM
        y0_px = (y0_centre_nm + fy * PIXEL_SIZE_NM) / PIXEL_SIZE_NM

        sigma_A_list.append(moment_sigma_nm(img_A, x0_px, y0_px, bg_rate_grid_main))
        sigma_B_list.append(moment_sigma_nm(img_B, x0_px, y0_px, bg_rate_grid_main))
        sigma_C_list.append(moment_sigma_nm(img_C, x0_px, y0_px, bg_rate_grid_main))

        total_A = img_A.sum(axis=(1, 2)).astype(float)
        total_B = img_B.sum(axis=(1, 2)).astype(float)
        total_C = img_C.sum(axis=(1, 2)).astype(float)
        N_A_list.append(total_A)
        N_B_list.append(total_B)
        N_C_list.append(total_C)

        frac_A_list.append((img_A * (colour_grid == dom_idx)).sum(axis=(1, 2)) / np.maximum(total_A, 1))
        frac_B_list.append((img_B * (colour_grid == dom_idx)).sum(axis=(1, 2)) / np.maximum(total_B, 1))
        frac_C_list.append((img_C * (colour_grid == dom_idx)).sum(axis=(1, 2)) / np.maximum(total_C, 1))

    metrics[dye] = {
        'sigma_A': np.concatenate(sigma_A_list), 'sigma_B': np.concatenate(sigma_B_list), 'sigma_C': np.concatenate(sigma_C_list),
        'N_A': np.concatenate(N_A_list), 'N_B': np.concatenate(N_B_list), 'N_C': np.concatenate(N_C_list),
        'frac_A': np.concatenate(frac_A_list), 'frac_B': np.concatenate(frac_B_list), 'frac_C': np.concatenate(frac_C_list),
        'channel': dom_channel,
    }

print(f"pooled {len(offsets_frac) * N_REPEATS} draws per model per dye (n_photon={mid_photon:.0f})")
for dye in dyes:
    print(f"  {dye}: dominant channel = {metrics[dye]['channel']}")

In [ ]:
# --- Figure: histogram (A/B/C overlaid) | ECDF-difference vs (C) (+ KS statistics) ---
# One figure per dye (3 rows x 2 cols) rather than one 9-row figure -- keeps each well within
# the publication-plotting framework's max figure depth (8.25"), and each dye's comparison is
# independently readable.
metric_specs = [
    ('sigma_PSF', 'sigma_A', 'sigma_B', 'sigma_C', 'sigma_PSF / nm'),
    ('N_photoelectrons', 'N_A', 'N_B', 'N_C', 'N_pe'),
    ('spectral fingerprint', 'frac_A', 'frac_B', 'frac_C', 'dominant-channel fraction'),
]

ks_rows = []
for dye in dyes:
    m = metrics[dye]
    fig, axs = plotter.two_column_plot(nrows=len(metric_specs), ncols=2, width=6.69, height=2.2 * len(metric_specs))

    for row, (name, key_A, key_B, key_C, xlabel) in enumerate(metric_specs):
        a, b, c = m[key_A], m[key_B], m[key_C]
        ax_hist, ax_diff = axs[row, 0], axs[row, 1]
        label = xlabel if name != 'spectral fingerprint' else f"{xlabel} ({m['channel']})"

        lo, hi = min(a.min(), b.min(), c.min()), max(a.max(), b.max(), c.max())
        bins = np.linspace(lo, hi, 40)

        ax_hist.hist(a, bins=bins, alpha=0.45, color='tab:blue', density=True, label='(A) factorised')
        ax_hist.hist(b, bins=bins, alpha=0.35, color='tab:orange', density=True, label='(B) photon-by-photon')
        ax_hist.hist(c, bins=bins, density=True, histtype='step', lw=1.5, color='0.2', label='(C) diffraction')
        ax_hist.axvline(a.mean(), color='tab:blue', ls=':', lw=1.3)
        ax_hist.axvline(b.mean(), color='tab:orange', ls=':', lw=1.3)
        ax_hist.axvline(c.mean(), color='0.2', ls=':', lw=1.3)
        ax_hist.set_xlabel(label, fontsize=7)
        ax_hist.set_ylabel('density', fontsize=7)
        ax_hist.tick_params(labelsize=6)
        ax_hist.set_title(name, fontsize=7)
        if row == 0:
            ax_hist.legend(fontsize=6, loc='upper right')

        ks_AC, ks_p_AC = ks_2samp(a, c)
        ks_BC, ks_p_BC = ks_2samp(b, c)
        grid = np.linspace(lo, hi, 400)
        ecdf_a = np.searchsorted(np.sort(a), grid, side='right') / len(a)
        ecdf_b = np.searchsorted(np.sort(b), grid, side='right') / len(b)
        ecdf_c = np.searchsorted(np.sort(c), grid, side='right') / len(c)
        ax_diff.plot(grid, (ecdf_a - ecdf_c) * 100, color='tab:blue', lw=1, label='A - C')
        ax_diff.plot(grid, (ecdf_b - ecdf_c) * 100, color='tab:orange', lw=1, label='B - C')
        ax_diff.axhline(0, color='0.6', lw=0.6)
        pct_diff_AC = 100 * (a.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
        pct_diff_BC = 100 * (b.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
        ax_diff.set_title(f"A-C: KS={ks_AC:.3f} (p={ks_p_AC:.2f}), {pct_diff_AC:+.2f}%   "
                           f"B-C: KS={ks_BC:.3f} (p={ks_p_BC:.2f}), {pct_diff_BC:+.2f}%", fontsize=6)
        ax_diff.set_xlabel(label, fontsize=7)
        ax_diff.set_ylabel('ECDF diff vs (C) / %', fontsize=7)
        ax_diff.tick_params(labelsize=6)
        if row == 0:
            ax_diff.legend(fontsize=6, loc='upper right')

        ks_rows.append({'dye': dye, 'metric': name,
                         'ks_AvsC': ks_AC, 'ks_p_AvsC': ks_p_AC, 'ks_BvsC': ks_BC, 'ks_p_BvsC': ks_p_BC,
                         'mean_A': a.mean(), 'mean_B': b.mean(), 'mean_C': c.mean(),
                         'pct_diff_AvsC': pct_diff_AC, 'pct_diff_BvsC': pct_diff_BC})

    fig.suptitle(dye, fontsize=8, y=1.02)
    plt.savefig(OUT_ROOT / f"distributional_comparison_{dye.replace(' ', '-')}.svg", bbox_inches='tight')
    plt.show()

ks_df = pd.DataFrame(ks_rows)
ks_df.to_csv(OUT_ROOT / "ks_comparison.csv", index=False)
ks_df

## Photon-count dependence: is the factorised model's error vs. the true optics worse at low or high photon counts?

The distributional comparison above only ran at the single middle photon level -- informative
about distribution *shape*, but a single level can't say whether that's close to the worst
case or not. Repeats the same per-repeat metric extraction (`sigma_PSF`, `N_photoelectrons`,
dominant-channel fraction) across *every* level in `photon_levels`, pooling the 3 offsets at
each level, to characterise (A) factorised's and (B) photon-by-photon's error against (C) the
true diffraction reference as a function of photon budget, rather than assuming it from one
point.

In [ ]:
# --- Same per-repeat metric extraction as above, generalised across ALL photon levels ---
def extract_metrics(dye, n_photon, results=results, offsets_frac=offsets_frac,
                     bg_rate_grid=bg_rate_grid_main, x0_centre_nm=x0_centre_nm,
                     y0_centre_nm=y0_centre_nm, pixel_order_indices=pixel_order_indices,
                     colour_grid=colour_grid, PIXEL_SIZE_NM=PIXEL_SIZE_NM):
    key0 = (dye, float(n_photon), offsets_frac[0])
    img_A0 = results[key0]['A']
    ch_means = {c: (img_A0 * (colour_grid == idx)).sum(axis=(1, 2)).mean()
                for c, idx in pixel_order_indices.items()}
    dom_channel = max(ch_means, key=ch_means.get)
    dom_idx = pixel_order_indices[dom_channel]

    sigma_A_list, sigma_B_list, sigma_C_list = [], [], []
    N_A_list, N_B_list, N_C_list = [], [], []
    frac_A_list, frac_B_list, frac_C_list = [], [], []

    for (fx, fy) in offsets_frac:
        key = (dye, float(n_photon), (fx, fy))
        img_A, img_B, img_C = results[key]['A'], results[key]['B'], results[key]['C']
        x0_px = (x0_centre_nm + fx * PIXEL_SIZE_NM) / PIXEL_SIZE_NM
        y0_px = (y0_centre_nm + fy * PIXEL_SIZE_NM) / PIXEL_SIZE_NM

        sigma_A_list.append(moment_sigma_nm(img_A, x0_px, y0_px, bg_rate_grid))
        sigma_B_list.append(moment_sigma_nm(img_B, x0_px, y0_px, bg_rate_grid))
        sigma_C_list.append(moment_sigma_nm(img_C, x0_px, y0_px, bg_rate_grid))

        total_A = img_A.sum(axis=(1, 2)).astype(float)
        total_B = img_B.sum(axis=(1, 2)).astype(float)
        total_C = img_C.sum(axis=(1, 2)).astype(float)
        N_A_list.append(total_A)
        N_B_list.append(total_B)
        N_C_list.append(total_C)

        frac_A_list.append((img_A * (colour_grid == dom_idx)).sum(axis=(1, 2)) / np.maximum(total_A, 1))
        frac_B_list.append((img_B * (colour_grid == dom_idx)).sum(axis=(1, 2)) / np.maximum(total_B, 1))
        frac_C_list.append((img_C * (colour_grid == dom_idx)).sum(axis=(1, 2)) / np.maximum(total_C, 1))

    return {
        'sigma_A': np.concatenate(sigma_A_list), 'sigma_B': np.concatenate(sigma_B_list), 'sigma_C': np.concatenate(sigma_C_list),
        'N_A': np.concatenate(N_A_list), 'N_B': np.concatenate(N_B_list), 'N_C': np.concatenate(N_C_list),
        'frac_A': np.concatenate(frac_A_list), 'frac_B': np.concatenate(frac_B_list), 'frac_C': np.concatenate(frac_C_list),
        'channel': dom_channel,
    }


metric_specs_all = [
    ('sigma_PSF', 'sigma_A', 'sigma_B', 'sigma_C'),
    ('N_photoelectrons', 'N_A', 'N_B', 'N_C'),
    ('spectral fingerprint', 'frac_A', 'frac_B', 'frac_C'),
]

level_rows = []
for dye in dyes:
    for n_photon in photon_levels:
        m = extract_metrics(dye, n_photon)
        for name, key_A, key_B, key_C in metric_specs_all:
            a, b, c = m[key_A], m[key_B], m[key_C]
            ks_AC, ks_p_AC = ks_2samp(a, c)
            ks_BC, ks_p_BC = ks_2samp(b, c)
            pct_diff_AC = 100 * (a.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
            pct_diff_BC = 100 * (b.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
            level_rows.append({
                'dye': dye, 'n_photon': n_photon, 'metric': name,
                'mean_A': a.mean(), 'mean_B': b.mean(), 'mean_C': c.mean(),
                'pct_diff_AvsC': pct_diff_AC, 'pct_diff_BvsC': pct_diff_BC,
                'ks_AvsC': ks_AC, 'ks_p_AvsC': ks_p_AC, 'ks_BvsC': ks_BC, 'ks_p_BvsC': ks_p_BC,
            })

level_df = pd.DataFrame(level_rows)
level_df.to_csv(OUT_ROOT / "pct_mean_diff_vs_photon_count.csv", index=False)
worst_AC = level_df.loc[level_df['pct_diff_AvsC'].abs().idxmax()]
worst_BC = level_df.loc[level_df['pct_diff_BvsC'].abs().idxmax()]
print(f"worst |A vs C| (factorised vs true diffraction): {worst_AC['pct_diff_AvsC']:.3f}% "
      f"at n_photon={worst_AC['n_photon']:.0f}, metric={worst_AC['metric']}, dye={worst_AC['dye']}")
print(f"worst |B vs C| (photon-by-photon Gaussian vs true diffraction): {worst_BC['pct_diff_BvsC']:.3f}% "
      f"at n_photon={worst_BC['n_photon']:.0f}, metric={worst_BC['metric']}, dye={worst_BC['dye']}")
level_df.pivot_table(index=['metric', 'n_photon'], columns='dye', values='pct_diff_AvsC').round(3)

In [ ]:
# --- Headline figure: does the production factorised model (A) stay close to the true
# diffraction-based optics reference (C)? Top row: absolute metric value vs photon count,
# (A) solid markers, (C) dotted -- do the lines sit on top of each other? Bottom row: percent
# deviation of (A) from (C), with a shaded +/-5% band as a visual "not too bad" reference.
fig, axs = plotter.two_column_plot(nrows=2, ncols=3, width=6.69, height=4.4)

dye_colours = {'ATTO 488': 'tab:blue', 'ATTO 565': 'tab:green', 'ATTO 647N': 'tab:red'}
metric_labels = {
    'sigma_PSF': r'$\sigma_{\rm PSF}$ / nm',
    'N_photoelectrons': r'$N_{\rm pe}$',
    'spectral fingerprint': 'dominant-channel fraction',
}

for col, (name, _, _, _) in enumerate(metric_specs_all):
    sub = level_df[level_df.metric == name]
    ax_val, ax_pct = axs[0, col], axs[1, col]

    for dye in dyes:
        d = sub[sub.dye == dye].sort_values('n_photon')
        ax_val.plot(d['n_photon'], d['mean_C'], lw=2.2, ls=':', color=dye_colours[dye], alpha=0.6,
                    label=f'{dye} — (C) true diffraction' if col == 0 else None)
        ax_val.plot(d['n_photon'], d['mean_A'], marker='o', ms=3, lw=1.2, ls='-', color=dye_colours[dye],
                    label=f'{dye} — (A) factorised (production)' if col == 0 else None)
        ax_pct.plot(d['n_photon'], d['pct_diff_AvsC'], marker='o', ms=3, lw=1.2, color=dye_colours[dye])

    ax_val.set_xscale('log')
    ax_val.set_xlabel('photons / punctum', fontsize=8)
    ax_val.set_ylabel(metric_labels[name], fontsize=8)
    ax_val.set_title(name, fontsize=8)
    ax_val.tick_params(labelsize=7)

    ax_pct.axhspan(-5, 5, color='0.85', zorder=0)
    ax_pct.axhline(0, color='0.5', lw=0.7)
    ax_pct.set_xscale('log')
    ax_pct.set_xlabel('photons / punctum', fontsize=8)
    ax_pct.set_ylabel('(A - C) / C  %', fontsize=8)
    ax_pct.tick_params(labelsize=7)

axs[0, 0].legend(fontsize=5, loc='best')
fig.suptitle('Production factorised model (A) vs. the true diffraction-based reference (C)', fontsize=8, y=1.02)
plt.savefig(OUT_ROOT / "model_comparison_vs_photon_count.svg", bbox_inches='tight')
plt.show()

**Conclusion:** the result splits cleanly along what each metric actually measures.
`N_photoelectrons` and the spectral fingerprint — both about total detected counts/colour, not
PSF shape — track (C) tightly at every photon level (well under 5%, see the figure above and
`pct_mean_diff_vs_photon_count.csv`): (A)'s spectrum-averaged QE step is not a meaningful
source of error against the true optics.

`sigma_PSF`, however, does **not** stay close, and the gap grows with photon count (see the
printed "worst |A vs C|" line above — tens of percent at the high end of the standard range).
This is *not* evidence that (A)'s Gaussian PSF width is badly wrong; it's a known weakness of
the fast moment-based `sigma_PSF` proxy used throughout this notebook, not the real nonlinear
Gaussian fit the production pipeline actually reports (see "Shared helpers" above). Checked
directly: the raw, noise-free `VectorialPSF` patch for ATTO 488 at 520 nm has genuine Airy
side-lobes (radial-profile bumps at r=4–5 and r=7–8 px, still-visible energy out to the 10-px
ROI edge) that push its own second-moment width to ~1.85x the matched Gaussian sigma — a
photon-weighted second moment is highly sensitive to faint distant tails that a Gaussian model
has none of by construction. `moment_sigma_nm`'s background-subtraction clipping hides part of
this at low photon counts (the faint rings sit below the noise floor and get clipped to zero)
and reveals more of it as photon count rises — exactly the growing-with-N pattern seen here.

**What this means for the reviewer response:** the intensity/colour side of the factorisation
(A) is validated directly against the true diffraction-based optics (C) and is not a
significant source of error. The PSF-width side cannot yet be validated the same way with this
proxy metric — doing that properly needs the real nonlinear Gaussian-fit sigma (production's
actual reported quantity) run on (A)'s and (C)'s images, which removes the tail-sensitivity
artifact but is considerably more expensive; see "Next steps" below.

## Next steps

- **Replace the moment-based `sigma_PSF` proxy with a real nonlinear Gaussian fit** (the
  quantity the production pipeline actually reports) for models (A)/(B)/(C) — this notebook's
  biggest current limitation, and now the main open question for the reviewer response given
  the conclusion above. The moment proxy is demonstrably tail-sensitive (see the diffraction-
  patch diagnostic in the conclusion); a Gaussian MLE/LSQ fit to each repeat's image would be a
  fairer way to ask whether the production PSF-width estimate changes once wavelength and true
  diffraction are treated correctly, since a fit isn't dragged around by faint Airy side-lobes
  the way a raw second moment is. Cost: one nonlinear fit per (dye, photon_level, offset,
  repeat, model) — likely needs a reduced `N_REPEATS`/`n_photon_levels` grid even in
  `FAST_MODE`, or a subsample of repeats per condition.
- **Full `FAST_MODE = False` run** for the published numbers, once the above is in place.
- Check whether `N_WAVELENGTH_BINS` in model (C) needs to be larger than the current defaults
  (8 for `FAST_MODE`, 15 for the full run, matching `N_PHOTON_STEPS`) for convergence — not yet
  checked against, e.g., 30 bins.